In [ ]:
# Cell 1: Title and Google Drive mount
from IPython.display import display, Markdown

display(Markdown('### Cell 1: Frozen VAE + PCA + PC labeling workflow'))
display(Markdown('This notebook reuses an existing trained VAE checkpoint. It intentionally contains no training loop.'))
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped outside Colab:', repr(e))


In [ ]:
# Cell 2: Repository, frozen manifest, imports, and output folders
display(Markdown('### Cell 2: Repository, frozen manifest, imports, and output folders'))
from pathlib import Path
import csv, json, os, pickle, shutil, subprocess, sys
import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.stats import spearmanr

REPO_URL = 'https://github.com/cindy-77jiayi/thesis_hapticAE.git'
PROJECT_ROOT = Path('/content/thesis_hapticAE')
INSTALL_REQUIREMENTS = True

if not (PROJECT_ROOT / 'src').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'], check=False)

requirements_path = PROJECT_ROOT / 'requirements.txt'
if INSTALL_REQUIREMENTS and requirements_path.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)], check=True)

sys.path.insert(0, str(PROJECT_ROOT))

DRIVE_ROOT = Path('/content/drive/MyDrive')
FROZEN_ROOT = DRIVE_ROOT / 'thesis' / 'frozen_model_outputs'
FROZEN_MANIFEST_PATH = FROZEN_ROOT / 'latest_frozen_manifest.json'
if not FROZEN_MANIFEST_PATH.exists():
    FROZEN_MANIFEST_PATH = FROZEN_ROOT / 'vae_balanced' / 'frozen_manifest.json'
if not FROZEN_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'Frozen manifest not found: {FROZEN_MANIFEST_PATH}. Run train_colab_freeze_to_drive.ipynb first.')

FROZEN_MANIFEST = json.loads(FROZEN_MANIFEST_PATH.read_text(encoding='utf-8'))
FROZEN_RUN_DIR = Path(FROZEN_MANIFEST.get('frozen_run_dir', FROZEN_MANIFEST_PATH.parent))
CONFIG_PATH = Path(FROZEN_MANIFEST['config_path'])
CHECKPOINT_PATH = Path(FROZEN_MANIFEST['checkpoint_path'])
FROZEN_PCA_DIR = Path(FROZEN_MANIFEST.get('pca_dir') or (FROZEN_RUN_DIR / 'pca'))
FROZEN_CONTROLS_DIR = Path(FROZEN_MANIFEST.get('controls_dir') or (FROZEN_RUN_DIR / 'controls'))

# Only needed if you intentionally force latent re-extraction. Default workflow uses frozen Drive artifacts.
DATA_DIR = Path('/content/hapticgen-dataset')
if not DATA_DIR.exists():
    DATA_DIR = PROJECT_ROOT / 'hapticgen-dataset'

ANALYSIS_NAME = 'frozen_vae_pca_workflow'
OUTPUT_DIR = FROZEN_RUN_DIR / 'analysis' / ANALYSIS_NAME
LATENT_DIR = OUTPUT_DIR / 'latent_cache'
PCA_DIR = OUTPUT_DIR / 'pca_cache'
SWEEP_DIR = OUTPUT_DIR / 'sweeps'
SUMMARY_DIR = OUTPUT_DIR / 'summary'
LLM_DIR = OUTPUT_DIR / 'llm_stub'
for d in [OUTPUT_DIR, LATENT_DIR, PCA_DIR, SWEEP_DIR, SUMMARY_DIR, LLM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

from src.utils.config import load_config
from src.utils.seed import set_seed
from src.data.loaders import build_dataloaders, build_model, load_checkpoint
from src.pipelines.latent_extraction import extract_latent_vectors
from src.pipelines.pca_control import fit_pca_pipeline, sweep_axis, control_to_latent

print('Frozen manifest:', FROZEN_MANIFEST_PATH)
print('Frozen run dir:', FROZEN_RUN_DIR)
print('CONFIG_PATH:', CONFIG_PATH, 'exists=', CONFIG_PATH.exists())
print('CHECKPOINT_PATH:', CHECKPOINT_PATH, 'exists=', CHECKPOINT_PATH.exists())
print('FROZEN_PCA_DIR:', FROZEN_PCA_DIR, 'exists=', FROZEN_PCA_DIR.exists())
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Frozen quality metrics:', json.dumps(FROZEN_MANIFEST.get('quality_metrics', {}), indent=2))


In [ ]:
# Cell 3: Load config and frozen VAE checkpoint
display(Markdown('### Cell 3: Load config and frozen VAE checkpoint'))
CONFIG = load_config(str(CONFIG_PATH))
set_seed(CONFIG.get('seed', 42))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if CONFIG.get('model_type', 'vae') != 'vae':
    raise ValueError('Selected config is not model_type=vae.')
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Frozen checkpoint not found: {CHECKPOINT_PATH}')

def load_frozen_vae(config, checkpoint_path, device):
    # Architecture is instantiated only as a container for checkpoint weights. No training is run here.
    model = build_model(config, device)
    try:
        model = load_checkpoint(model, str(checkpoint_path), device)
    except Exception as e:
        print('Default loader failed; trying wrapped checkpoint:', repr(e))
        try:
            state = torch.load(str(checkpoint_path), map_location=device, weights_only=True)
        except TypeError:
            state = torch.load(str(checkpoint_path), map_location=device)
        if isinstance(state, dict):
            for key in ['model_state_dict', 'state_dict', 'model']:
                if key in state and isinstance(state[key], dict):
                    state = state[key]
                    break
        if isinstance(state, dict) and all(str(k).startswith('module.') for k in state.keys()):
            state = {str(k).replace('module.', '', 1): v for k, v in state.items()}
        model.load_state_dict(state)
        model = model.to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model

model = load_frozen_vae(CONFIG, CHECKPOINT_PATH, DEVICE)
assert model.training is False
assert sum(p.numel() for p in model.parameters() if p.requires_grad) == 0
print('Loaded frozen VAE:', CHECKPOINT_PATH)
print('Device:', DEVICE)
print('Run name:', CONFIG.get('run_name'))
print('Latent dim:', CONFIG['model']['latent_dim'])
print('Frozen quality score:', FROZEN_MANIFEST.get('quality_score'))
print('Frozen PCA coverage:', FROZEN_MANIFEST.get('pca_coverage'))


In [ ]:
# Cell 4: Load frozen latent cache or optionally re-extract mu
display(Markdown('### Cell 4: Frozen latent cache'))
LATENT_KIND = 'mu'  # Recommended: deterministic mu. Use sampled z only for a deliberate experiment.
BATCH_SIZE = 64
FORCE_REEXTRACT_LATENTS = False

frozen_latent_path = Path(FROZEN_MANIFEST.get('latent_path') or (FROZEN_PCA_DIR / 'Z_mu.npy'))
frozen_ids_json_path = Path(FROZEN_MANIFEST.get('sample_ids_path') or (FROZEN_PCA_DIR / 'sample_ids.json'))
latent_path = frozen_latent_path
ids_json_path = frozen_ids_json_path
ids_csv_path = LATENT_DIR / 'sample_ids.csv'

if frozen_latent_path.exists() and frozen_ids_json_path.exists() and not FORCE_REEXTRACT_LATENTS:
    Z = np.load(frozen_latent_path).astype(np.float32)
    sample_ids = json.loads(frozen_ids_json_path.read_text(encoding='utf-8'))
    print('Loaded frozen latent cache:', frozen_latent_path, Z.shape)
else:
    if not DATA_DIR.exists():
        raise FileNotFoundError(f'DATA_DIR is required only for re-extraction, but it was not found: {DATA_DIR}')
    data = build_dataloaders(CONFIG, str(DATA_DIR), batch_size=BATCH_SIZE, full_dataset=True)
    sample_ids = [os.path.relpath(p, str(DATA_DIR)).replace(os.sep, '/') for p in data['wav_files']]
    latent_path = LATENT_DIR / f'Z_{LATENT_KIND}.npy'
    ids_json_path = LATENT_DIR / 'sample_ids.json'

    def extract_latents(kind):
        if kind == 'mu':
            return extract_latent_vectors(model, data['all_loader'], DEVICE)
        if kind != 'z':
            raise ValueError("LATENT_KIND must be 'mu' or 'z'.")
        from tqdm.auto import tqdm
        model.eval(); all_z=[]
        with torch.no_grad():
            for x in tqdm(data['all_loader'], desc='Extracting sampled z', leave=False):
                z, _, _ = model.encode(x.to(DEVICE))
                all_z.append(z.cpu().numpy())
        return np.concatenate(all_z, axis=0)

    Z = extract_latents(LATENT_KIND).astype(np.float32)
    np.save(latent_path, Z)
    ids_json_path.write_text(json.dumps(sample_ids, indent=2), encoding='utf-8')
    print('Saved re-extracted latent cache:', latent_path, Z.shape)

with open(ids_csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['row_index', 'sample_id'])
    w.writerows([[i, sid] for i, sid in enumerate(sample_ids)])

assert Z.shape[0] == len(sample_ids)
assert Z.shape[1] == CONFIG['model']['latent_dim']
print('Sample ids:', len(sample_ids))
print('Sample ids CSV for analysis:', ids_csv_path)


In [ ]:
# Cell 5: Load frozen PCA artifacts or optionally refit PCA
display(Markdown('### Cell 5: Frozen PCA artifacts'))
N_COMPONENTS = 8
FORCE_REFIT_PCA = False

frozen_pca_pipe_path = Path(FROZEN_MANIFEST.get('pca_pipe_path') or (FROZEN_PCA_DIR / 'pca_pipe.pkl'))
frozen_zpca_path = FROZEN_PCA_DIR / 'Z_pca.npy'
frozen_pca_artifact_path = Path(FROZEN_MANIFEST.get('pca_artifacts_path') or (FROZEN_PCA_DIR / 'pca_artifacts.npz'))

if frozen_pca_pipe_path.exists() and frozen_zpca_path.exists() and not FORCE_REFIT_PCA:
    pca_pipe_path = frozen_pca_pipe_path
    zpca_path = frozen_zpca_path
    pca_artifact_path = frozen_pca_artifact_path
    with open(pca_pipe_path, 'rb') as f:
        pipe = pickle.load(f)
    Z_pca = np.load(zpca_path).astype(np.float32)
    print('Loaded frozen PCA:', pca_pipe_path, Z_pca.shape)
else:
    pca_pipe_path = PCA_DIR / 'pca_pipe.pkl'
    zpca_path = PCA_DIR / 'Z_pca.npy'
    pca_artifact_path = PCA_DIR / 'pca_artifacts.npz'
    pipe, Z_pca = fit_pca_pipeline(Z, n_components=N_COMPONENTS, save_dir=str(PCA_DIR))
    print('Refit PCA into analysis cache:', PCA_DIR)

pca = pipe.named_steps['pca']
scaler = pipe.named_steps['scaler']
N_COMPONENTS = int(pca.n_components_ if hasattr(pca, 'n_components_') else pca.n_components)
pca_coverage = float(pca.explained_variance_ratio_.sum())

analysis_pca_artifact_path = PCA_DIR / 'pca_artifacts.npz'
np.savez(
    analysis_pca_artifact_path,
    components=pca.components_.astype(np.float32),
    explained_variance=pca.explained_variance_.astype(np.float32),
    explained_variance_ratio=pca.explained_variance_ratio_.astype(np.float32),
    pca_mean=pca.mean_.astype(np.float32),
    scaler_mean=scaler.mean_.astype(np.float32),
    scaler_scale=scaler.scale_.astype(np.float32),
    Z_pca=Z_pca.astype(np.float32),
)

print('PCA source artifact:', pca_artifact_path, 'exists=', pca_artifact_path.exists())
print('PCA analysis artifact copy:', analysis_pca_artifact_path)
print('Explained variance ratio:', np.round(pca.explained_variance_ratio_, 4))
print('Total PCA coverage:', f'{pca_coverage:.2%}')
print('For your frozen run, focus first on PC1-PC3; treat PC6-PC8 as tentative unless Cell 8 says strong and non-duplicate.')


In [ ]:
# Cell 6: Sweep range modes and multi-anchor references
display(Markdown('### Cell 6: Sweep ranges and multi-anchor setup'))
SWEEP_RANGE_MODE = 'robust_quantile_range'  # fixed_range, empirical_minmax, robust_quantile_range
FIXED_RANGE = (-1.0, 1.0)
ROBUST_QUANTILES = (5.0, 95.0)
N_SWEEP_STEPS = 21
N_ANCHORS = 5
PC_AXES_TO_SWEEP = list(range(min(8, Z_pca.shape[1])))

# Metrics with high reconstruction error in the latest eval should be interpreted more cautiously during labeling.
CAUTION_LABEL_METRICS = ['attack_time_s', 'onset_density_ps', 'envelope_decay_slope_dBps', 'short_term_variance', 'spectral_flatness']
PRIORITY_LABEL_METRICS = ['rms_energy', 'crest_factor', 'spectral_centroid_hz', 'envelope_entropy_bits', 'ioi_entropy_bits', 'am_modulation_index']

def compute_sweep_ranges(Z_pca, mode):
    ranges=[]
    for axis in PC_AXES_TO_SWEEP:
        col=Z_pca[:, axis]
        qlo,qhi=np.percentile(col, ROBUST_QUANTILES)
        if mode == 'fixed_range': lo,hi = FIXED_RANGE
        elif mode == 'empirical_minmax': lo,hi = float(col.min()), float(col.max())
        elif mode == 'robust_quantile_range': lo,hi = float(qlo), float(qhi)
        else: raise ValueError(f'Unknown range mode: {mode}')
        ranges.append({'axis':axis,'pc':f'PC{axis+1}','mode':mode,'low':round(float(lo),6),'high':round(float(hi),6),'mean':round(float(col.mean()),6),'std':round(float(col.std()),6),'min':round(float(col.min()),6),'max':round(float(col.max()),6),'q_low':round(float(qlo),6),'q_high':round(float(qhi),6)})
    return ranges

def choose_anchor_indices(Z_pca, n_anchors):
    # Use central-to-moderate real samples rather than extreme outliers.
    scores=np.linalg.norm(Z_pca[:, :min(3, Z_pca.shape[1])], axis=1)
    out=[]
    for q in np.linspace(0.2, 0.8, n_anchors):
        idx=int(np.argmin(np.abs(scores - np.quantile(scores, q))))
        if idx not in out: out.append(idx)
    rng=np.random.default_rng(CONFIG.get('seed', 42))
    while len(out) < n_anchors:
        idx=int(rng.integers(0, Z_pca.shape[0]))
        if idx not in out: out.append(idx)
    return out[:n_anchors]

sweep_ranges = compute_sweep_ranges(Z_pca, SWEEP_RANGE_MODE)
anchor_indices = choose_anchor_indices(Z_pca, N_ANCHORS)
anchor_refs = Z_pca[anchor_indices].astype(np.float32)
(SWEEP_DIR / 'sweep_ranges.json').write_text(json.dumps(sweep_ranges, indent=2), encoding='utf-8')
(SWEEP_DIR / 'anchor_samples.json').write_text(json.dumps([{'anchor_id':i,'row_index':int(idx),'sample_id':sample_ids[idx]} for i,idx in enumerate(anchor_indices)], indent=2), encoding='utf-8')
print('Range mode:', SWEEP_RANGE_MODE)
print('PCA coverage:', f'{pca_coverage:.2%}')
print('Sweeping PCs:', [r['pc'] for r in sweep_ranges])
print('Anchors:', anchor_indices)
print('Priority metrics for labels:', PRIORITY_LABEL_METRICS)
print('Caution metrics:', CAUTION_LABEL_METRICS)
for r in sweep_ranges:
    print(f"{r['pc']}: [{r['low']:+.3f}, {r['high']:+.3f}]")


In [ ]:
# Cell 7: Single-axis sweep visualization and audio playback
display(Markdown('### Cell 7: Single-axis sweep visualization + audio playback'))
from IPython.display import Audio
from src.pipelines.pca_control import plot_sweep, play_sweep

SINGLE_AXIS_PC_AXES = [0, 1, 2, 3]  # PC1-PC4 first. Add 4-7 only after these are understood.
SINGLE_AXIS_RANGE_MODE = 'robust_quantile_range'  # robust_quantile_range or fixed_pm2
SINGLE_AXIS_FIXED_RANGE = (-2.0, 2.0)
SINGLE_AXIS_N_STEPS = 9
SINGLE_AXIS_REFERENCE_MODE = 'anchor'  # anchor or origin
SINGLE_AXIS_ANCHOR_ID = 0
SINGLE_AXIS_PLAY_AUDIO = True
SINGLE_AXIS_SAVE_SIGNALS = True

SINGLE_AXIS_DIR = OUTPUT_DIR / 'single_axis_sweeps'
SINGLE_AXIS_DIR.mkdir(parents=True, exist_ok=True)
sr = int(CONFIG['data']['sr'])
T = int(CONFIG['data']['T'])

def _audio_norm(x):
    x = np.asarray(x, dtype=np.float32)
    return np.clip(x / (np.max(np.abs(x)) + 1e-8), -1.0, 1.0)

def _range_for_axis(axis):
    if SINGLE_AXIS_RANGE_MODE == 'fixed_pm2':
        return SINGLE_AXIS_FIXED_RANGE
    if SINGLE_AXIS_RANGE_MODE == 'robust_quantile_range':
        match = [r for r in sweep_ranges if int(r['axis']) == int(axis)]
        if not match:
            raise ValueError(f'No robust range found for PC{axis + 1}')
        return (float(match[0]['low']), float(match[0]['high']))
    raise ValueError(f'Unknown SINGLE_AXIS_RANGE_MODE: {SINGLE_AXIS_RANGE_MODE}')

if SINGLE_AXIS_REFERENCE_MODE == 'origin':
    single_axis_reference = None
    reference_label = 'PCA origin'
else:
    single_axis_reference = anchor_refs[SINGLE_AXIS_ANCHOR_ID]
    anchor_row = int(anchor_indices[SINGLE_AXIS_ANCHOR_ID])
    reference_label = f"anchor {SINGLE_AXIS_ANCHOR_ID} | row {anchor_row} | {sample_ids[anchor_row]}"

print('Single-axis reference:', reference_label)
print('Single-axis range mode:', SINGLE_AXIS_RANGE_MODE)
print('PC axes:', [f'PC{i + 1}' for i in SINGLE_AXIS_PC_AXES])

# Optional original WAV playback when the dataset is available in this Colab runtime.
if SINGLE_AXIS_REFERENCE_MODE == 'anchor':
    original_path = Path(DATA_DIR) / sample_ids[anchor_row]
    if original_path.exists():
        try:
            import soundfile as sf
            original_signal, original_sr = sf.read(str(original_path), always_2d=False)
            if original_signal.ndim > 1:
                original_signal = original_signal[:, 0]
            display(Markdown(f'#### Original anchor WAV: `{sample_ids[anchor_row]}`'))
            display(Audio(_audio_norm(original_signal), rate=int(original_sr)))
        except Exception as e:
            print('Original WAV playback skipped:', repr(e))
    else:
        print('Original WAV not found in this runtime; decoded sweep playback will still work:', original_path)

# Decoded baseline for the selected PCA reference. This is the frozen VAE reconstruction used as the sweep center.
if single_axis_reference is not None:
    with torch.no_grad():
        z_ref = control_to_latent(pipe, single_axis_reference)
        ref_signal = model.decode(torch.from_numpy(z_ref).float().unsqueeze(0).to(DEVICE), target_len=T).squeeze().cpu().numpy()
    np.save(SINGLE_AXIS_DIR / 'reference_decoded_signal.npy', ref_signal.astype(np.float32))
    display(Markdown('#### Frozen VAE decoded reference / sweep center'))
    display(Audio(_audio_norm(ref_signal), rate=sr))

single_axis_payload = {
    'reference_mode': SINGLE_AXIS_REFERENCE_MODE,
    'reference_label': reference_label,
    'range_mode': SINGLE_AXIS_RANGE_MODE,
    'n_steps': SINGLE_AXIS_N_STEPS,
    'pc_axes': [int(a) for a in SINGLE_AXIS_PC_AXES],
    'results': {},
}

for axis in SINGLE_AXIS_PC_AXES:
    pc_name = f'PC{axis + 1}'
    this_range = _range_for_axis(axis)
    print(f"\n{'='*60}")
    print(f"Single-axis sweep {pc_name}: {this_range[0]:+.3f} to {this_range[1]:+.3f}")
    print(f"{'='*60}")
    result = sweep_axis(
        pipe,
        model,
        DEVICE,
        axis=axis,
        sweep_range=this_range,
        n_steps=SINGLE_AXIS_N_STEPS,
        T=T,
        sr=sr,
        reference=single_axis_reference,
        with_metrics=True,
    )
    plot_sweep(result, sr=sr, save_path=str(SINGLE_AXIS_DIR / f'{pc_name}_waveforms.png'))
    if SINGLE_AXIS_PLAY_AUDIO:
        play_sweep(result, sr=sr)
    if SINGLE_AXIS_SAVE_SIGNALS:
        np.savez(
            SINGLE_AXIS_DIR / f'{pc_name}_single_axis_sweep.npz',
            values=np.array(result['values'], dtype=np.float32),
            signals=result['signals'].astype(np.float32),
        )
    single_axis_payload['results'][pc_name] = {
        'axis': int(axis),
        'range': [float(this_range[0]), float(this_range[1])],
        'waveform_plot': str(SINGLE_AXIS_DIR / f'{pc_name}_waveforms.png'),
        'signals_npz': str(SINGLE_AXIS_DIR / f'{pc_name}_single_axis_sweep.npz'),
    }

(SINGLE_AXIS_DIR / 'single_axis_sweep_manifest.json').write_text(json.dumps(single_axis_payload, indent=2), encoding='utf-8')
print('Saved single-axis sweep outputs:', SINGLE_AXIS_DIR)
print('Next: use subjective listening here, then run the multi-anchor sweep and Cell 9 labeling to validate whether the perceived change is stable.')


In [ ]:
# Cell 8: Multi-anchor PC sweeps with metric cache
display(Markdown('### Cell 8: PC sweeping across multiple anchors'))
FORCE_RESWEEP = False
SAVE_SIGNAL_PREVIEWS = False
sweep_cache_path = SWEEP_DIR / f'sweep_metrics_{SWEEP_RANGE_MODE}_{N_ANCHORS}anchors_{N_SWEEP_STEPS}steps.json'

if sweep_cache_path.exists() and not FORCE_RESWEEP:
    sweep_payload = json.loads(sweep_cache_path.read_text(encoding='utf-8'))
    print('Loaded cached sweep metrics:', sweep_cache_path)
else:
    sweep_payload={'range_mode':SWEEP_RANGE_MODE,'n_sweep_steps':N_SWEEP_STEPS,'n_anchors':N_ANCHORS,'ranges':sweep_ranges,'anchor_indices':[int(i) for i in anchor_indices],'anchor_sample_ids':[sample_ids[i] for i in anchor_indices],'sweeps':{}}
    for r in sweep_ranges:
        axis=int(r['axis']); pc_key=f'PC{axis+1}'; sweep_payload['sweeps'][pc_key]=[]
        print(f"Sweeping {pc_key} on {N_ANCHORS} anchors")
        for aid,row_idx in enumerate(anchor_indices):
            sweep=sweep_axis(pipe, model, DEVICE, axis=axis, sweep_range=(r['low'], r['high']), n_steps=N_SWEEP_STEPS, T=CONFIG['data']['T'], sr=CONFIG['data']['sr'], reference=anchor_refs[aid], with_metrics=True)
            sweep_payload['sweeps'][pc_key].append({'anchor_id':int(aid),'row_index':int(row_idx),'sample_id':sample_ids[row_idx],'values':[float(v) for v in sweep['values']],'metrics':[{k:float(v) for k,v in m.items()} for m in sweep['metrics']]})
            if SAVE_SIGNAL_PREVIEWS and aid == 0:
                np.savez(SWEEP_DIR / f'{pc_key}_anchor0_signal_preview.npz', values=np.array(sweep['values'], dtype=np.float32), signals=sweep['signals'].astype(np.float32))
    sweep_cache_path.write_text(json.dumps(sweep_payload, indent=2), encoding='utf-8')
    print('Saved sweep metrics:', sweep_cache_path)
print('Sweep PCs:', list(sweep_payload['sweeps'].keys()))


In [ ]:
# Cell 9: Metric analysis, PC labeling, summary CSV, and plots
display(Markdown('### Cell 9: PC labeling analysis and summary outputs'))
METRIC_CONCEPTS={'rms_energy':'intensity','peak_amplitude':'intensity','crest_factor':'impulsiveness','spectral_centroid_hz':'brightness','spectral_rolloff_hz':'brightness','spectral_slope':'spectral tilt','spectral_flatness':'spectral texture','high_freq_ratio':'brightness','low_high_band_ratio':'warmth','band_energy_0_150':'low band energy','band_energy_150_400':'mid band energy','band_energy_400_800':'high band energy','envelope_decay_slope_dBps':'decay envelope','late_early_energy_ratio':'late energy','attack_time_s':'attack softness','transient_energy_ratio':'transient emphasis','effective_duration_s':'duration','envelope_area':'intensity','envelope_entropy_bits':'envelope complexity','onset_density_ps':'event density','ioi_entropy_bits':'rhythmic irregularity','onset_interval_cv':'rhythmic irregularity','modulation_peak_hz':'modulation rate','zero_crossing_rate_ps':'high-frequency texture','gap_ratio':'gappiness','short_term_variance':'energy variation','am_modulation_index':'amplitude modulation'}

def safe_spearman(x,y):
    y=np.asarray(y,dtype=float)
    if len(y) < 3 or np.std(y) < 1e-12: return 0.0,1.0
    r,p=spearmanr(x,y)
    return (0.0 if not np.isfinite(r) else float(r)), (1.0 if not np.isfinite(p) else float(p))

def rel_effect(y):
    y=np.asarray(y,dtype=float); return float((y.max()-y.min())/(np.mean(np.abs(y))+1e-10))

def summarize_pc(items):
    metric_names=list(items[0]['metrics'][0].keys()); per={}
    for mn in metric_names:
        rhos=[]; pvals=[]; effects=[]
        for item in items:
            vals=np.array(item['values'], dtype=float); ys=np.array([m[mn] for m in item['metrics']], dtype=float)
            r,p=safe_spearman(vals,ys); rhos.append(r); pvals.append(p); effects.append(rel_effect(ys))
        active=[np.sign(r) for r in rhos if abs(r)>=0.2]
        cons=0.0 if not active else max(sum(s>0 for s in active), sum(s<0 for s in active))/len(active)
        per[mn]={'mean_rho':float(np.mean(rhos)),'mean_abs_rho':float(np.mean(np.abs(rhos))),'mean_pvalue':float(np.mean(pvals)),'mean_effect':float(np.mean(effects)),'sign_consistency':float(cons)}
        per[mn]['score']=per[mn]['mean_abs_rho']*min(per[mn]['mean_effect'],2.0)*max(per[mn]['sign_consistency'],0.25)
    return {'metric_names':metric_names,'per_metric':per,'ranking':sorted(per.items(), key=lambda kv: kv[1]['score'], reverse=True)}

def label_pc(pc_i, top_metrics):
    usable=[(n,i) for n,i in top_metrics if n!='peak_amplitude']
    if not usable: return 'mixed / weak','weak'
    concepts=[]
    for name,_ in usable[:3]:
        c=METRIC_CONCEPTS.get(name, name.replace('_',' '))
        if c not in concepts: concepts.append(c)
    top_name, top = usable[0]
    caution_driven = top_name in CAUTION_LABEL_METRICS
    label=' / '.join(concepts[:2])
    strong=top['mean_abs_rho']>=0.70 and top['mean_effect']>=0.20 and top['sign_consistency']>=0.80
    moderate=top['mean_abs_rho']>=0.50 and top['mean_effect']>=0.12 and top['sign_consistency']>=0.65
    conf='strong' if strong else ('moderate' if moderate else 'weak')

    # Tail PCs are only promoted if they are very stable and not driven by metrics that reconstructed poorly.
    if pc_i>=4 and (conf!='strong' or caution_driven):
        return 'mixed / weak','weak'
    if conf=='weak':
        return 'mixed / weak','weak'
    if caution_driven and conf=='strong':
        conf='moderate'
    return label,conf

pc_details={pc:summarize_pc(items) for pc,items in sweep_payload['sweeps'].items()}
pcs=list(pc_details.keys()); metric_names=pc_details[pcs[0]]['metric_names']
sig=np.array([[pc_details[pc]['per_metric'][mn]['mean_rho'] for mn in metric_names] for pc in pcs], dtype=float)
normed=sig/(np.linalg.norm(sig,axis=1,keepdims=True)+1e-9); dup_sim=np.abs(normed @ normed.T)
summary_rows=[]
for i,pc in enumerate(pcs):
    sims=dup_sim[i].copy(); sims[i]=-1; j=int(np.argmax(sims)); dup=float(sims[j]); top=pc_details[pc]['ranking'][:5]
    label,conf=label_pc(i,top); top_name,top_info=top[0]
    dom='; '.join([f"{n}:{inf['mean_rho']:+.2f}/eff{inf['mean_effect']:.2f}/cons{inf['sign_consistency']:.2f}" for n,inf in top[:4]])
    summary_rows.append({'pc':pc,'axis':i,'explained_variance_ratio':round(float(pca.explained_variance_ratio_[i]),6),'dominant_metrics':dom,'top_metric':top_name,'monotonicity_abs_rho':round(float(top_info['mean_abs_rho']),4),'effect_size_relative':round(float(top_info['mean_effect']),4),'cross_anchor_consistency':round(float(top_info['sign_consistency']),4),'duplicate_of':'','duplicate_similarity':round(dup,4),'caution_metric_driven':bool(top_name in CAUTION_LABEL_METRICS),'final_label':label,'confidence':conf})

def _pc_preference(row):
    rank={'strong':2,'moderate':1,'weak':0}.get(row['confidence'],0)
    return (rank, float(row['explained_variance_ratio']), -int(row['axis']))

DUPLICATE_THRESHOLD = 0.85
for i in range(len(summary_rows)):
    for j in range(i + 1, len(summary_rows)):
        if dup_sim[i, j] < DUPLICATE_THRESHOLD:
            continue
        winner, loser = (i, j) if _pc_preference(summary_rows[i]) >= _pc_preference(summary_rows[j]) else (j, i)
        summary_rows[loser]['duplicate_of'] = summary_rows[winner]['pc']

for row in summary_rows:
    if row['duplicate_of'] or row['confidence']=='weak':
        row['use_recommendation']='disable'
    elif row['caution_metric_driven']:
        row['use_recommendation']='manual_review'
    else:
        row['use_recommendation']='use'

csv_path=SUMMARY_DIR/'pc_label_summary.csv'; json_path=SUMMARY_DIR/'pc_label_summary.json'; heatmap_path=SUMMARY_DIR/'pc_metric_monotonicity_heatmap.png'; dup_path=SUMMARY_DIR/'pc_duplicate_similarity.png'
with open(csv_path,'w',newline='',encoding='utf-8') as f:
    fields=list(summary_rows[0].keys()); w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(summary_rows)
json_path.write_text(json.dumps({'range_mode':SWEEP_RANGE_MODE,'n_anchors':N_ANCHORS,'n_sweep_steps':N_SWEEP_STEPS,'rows':summary_rows}, indent=2), encoding='utf-8')
try:
    import pandas as pd
    display(pd.DataFrame(summary_rows))
except Exception:
    print(json.dumps(summary_rows, indent=2))

fig,ax=plt.subplots(figsize=(max(14,len(metric_names)*0.55),5)); im=ax.imshow(sig,cmap='RdBu_r',vmin=-1,vmax=1,aspect='auto')
ax.set_yticks(range(len(pcs))); ax.set_yticklabels(pcs); ax.set_xticks(range(len(metric_names))); ax.set_xticklabels([m.replace('_',' ')[:22] for m in metric_names], rotation=65, ha='right', fontsize=8); ax.set_title('Mean Spearman rho across anchors'); fig.colorbar(im, ax=ax, shrink=0.8); fig.tight_layout(); fig.savefig(heatmap_path,dpi=160,bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(figsize=(6,5)); im=ax.imshow(dup_sim,cmap='viridis',vmin=0,vmax=1); ax.set_xticks(range(len(pcs))); ax.set_yticks(range(len(pcs))); ax.set_xticklabels(pcs); ax.set_yticklabels(pcs)
for i in range(len(pcs)):
    for j in range(len(pcs)):
        ax.text(j,i,f'{dup_sim[i,j]:.2f}',ha='center',va='center',color='white' if dup_sim[i,j]>0.55 else 'black',fontsize=8)
ax.set_title('Metric-signature similarity between PCs'); fig.colorbar(im, ax=ax, shrink=0.8); fig.tight_layout(); fig.savefig(dup_path,dpi=160,bbox_inches='tight'); plt.show()
print('Saved:', csv_path); print('Saved:', json_path); print('Saved:', heatmap_path); print('Saved:', dup_path)

guidance = f"""
### How to read these outputs
1. Open `pc_label_summary.csv` first. Use PCs where `confidence` is `strong` or `moderate` and `duplicate_of` is empty.
2. Inspect the heatmap next. A reliable PC should have a clear signed pattern across priority metrics, not random mixed signs.
3. Inspect the duplicate similarity plot. If two PCs have similarity >= 0.85, the notebook now keeps the stronger/higher-variance PC and disables the weaker duplicate.
4. For this frozen run, start with PC1-PC3 because PCA coverage is {pca_coverage:.2%} and the first three PCs explain most of the stable variation.
5. Treat PC6-PC8 as disabled unless Cell 9 marks them `use`; caution-metric-driven tail PCs are automatically held back.
6. Be cautious when labels depend mainly on: {', '.join(CAUTION_LABEL_METRICS)}.
"""
(SUMMARY_DIR/'how_to_read_pc_labels.md').write_text(guidance, encoding='utf-8')
display(Markdown(guidance))
print('Saved:', SUMMARY_DIR/'how_to_read_pc_labels.md')


In [ ]:
# Cell 10: LLM control stub and output manifest
display(Markdown('### Cell 10: LLM control stub and manifest'))
# Interface only: input text/semantic label plus optional explicit PC intensities in [0,1]; output PC weight vector.
ranges_by_pc={r['pc']:r for r in sweep_ranges}

def unit_to_pc_value(unit_value, pc_name):
    unit_value=float(np.clip(unit_value,0.0,1.0)); r=ranges_by_pc[pc_name]
    return float(r['low'] + unit_value * (r['high'] - r['low']))

def semantic_controls_to_pc_vector(semantic_controls, n_components=N_COMPONENTS):
    pc_vector=np.zeros(n_components,dtype=np.float32)
    for pc_name,unit_value in semantic_controls.items():
        if pc_name not in ranges_by_pc: raise ValueError(f'Unknown PC name: {pc_name}')
        pc_vector[int(ranges_by_pc[pc_name]['axis'])]=unit_to_pc_value(unit_value, pc_name)
    return pc_vector

def llm_text_to_pc_weights_stub(text_or_label, manual_semantic_controls=None):
    if manual_semantic_controls is None:
        pc_vector=np.zeros(N_COMPONENTS,dtype=np.float32); source='stub_zero_vector_until_llm_parser_is_added'
    else:
        pc_vector=semantic_controls_to_pc_vector(manual_semantic_controls); source='manual_semantic_controls'
    return {'input_text_or_label':text_or_label,'source':source,'pc_vector':[float(v) for v in pc_vector],'note':'Keep weak or duplicate PCs at 0 unless later validation marks them strong and distinct.'}

recommended=[r for r in summary_rows if r['confidence'] in ['strong','moderate'] and not r['duplicate_of']]
not_recommended=[r for r in summary_rows if r['confidence']=='weak' or r['duplicate_of']]
interface={
    'input':'text_or_label plus optional manual_semantic_controls such as {PC1: 0.8, PC2: 0.3}',
    'output':'pc_vector in PCA coordinates',
    'range_mode':SWEEP_RANGE_MODE,
    'recommended_pcs':recommended,
    'weak_or_duplicate_pcs':not_recommended,
    'priority_label_metrics':PRIORITY_LABEL_METRICS,
    'caution_label_metrics':CAUTION_LABEL_METRICS,
}
(LLM_DIR/'llm_control_interface_stub.json').write_text(json.dumps(interface,indent=2),encoding='utf-8')
example_controls = {'PC1':0.75}
if 'PC2' in ranges_by_pc:
    example_controls['PC2'] = 0.65
example=llm_text_to_pc_weights_stub('example: stronger and more irregular haptic feel', example_controls)
(LLM_DIR/'pc_vector_stub_example.json').write_text(json.dumps(example,indent=2),encoding='utf-8')

GENERATE_STUB_SIGNAL=False
if GENERATE_STUB_SIGNAL:
    pc_vector=np.array(example['pc_vector'],dtype=np.float32); z_np=control_to_latent(pipe,pc_vector); z_t=torch.from_numpy(z_np).float().unsqueeze(0).to(DEVICE)
    with torch.no_grad(): signal=model.decode(z_t,target_len=CONFIG['data']['T']).squeeze().cpu().numpy()
    np.save(LLM_DIR/'generated_haptic_stub.npy',signal.astype(np.float32))
    fig,ax=plt.subplots(figsize=(10,3)); t=np.arange(len(signal))/CONFIG['data']['sr']; ax.plot(t,signal,linewidth=0.8); ax.set_title('Frozen VAE decode from LLM stub PC vector'); ax.set_xlabel('Time (s)'); ax.set_ylabel('Amplitude'); fig.tight_layout(); fig.savefig(LLM_DIR/'generated_haptic_stub_preview.png',dpi=160,bbox_inches='tight'); plt.show()

manifest={
    'frozen_manifest':str(FROZEN_MANIFEST_PATH),
    'frozen_run_dir':str(FROZEN_RUN_DIR),
    'frozen_quality_score':FROZEN_MANIFEST.get('quality_score'),
    'frozen_quality_metrics':FROZEN_MANIFEST.get('quality_metrics', {}),
    'checkpoint':str(CHECKPOINT_PATH),
    'config':str(CONFIG_PATH),
    'latent_cache':str(latent_path),
    'sample_ids_json':str(ids_json_path),
    'sample_ids_csv':str(ids_csv_path),
    'pca_pipe':str(pca_pipe_path),
    'pca_scores':str(zpca_path),
    'pca_artifacts':str(pca_artifact_path),
    'analysis_pca_artifacts':str(analysis_pca_artifact_path),
    'pca_coverage':pca_coverage,
    'single_axis_sweep_dir':str(SINGLE_AXIS_DIR),
    'single_axis_sweep_manifest':str(SINGLE_AXIS_DIR/'single_axis_sweep_manifest.json'),
    'single_axis_reference_decoded_signal':str(SINGLE_AXIS_DIR/'reference_decoded_signal.npy'),
    'sweep_ranges':str(SWEEP_DIR/'sweep_ranges.json'),
    'anchor_samples':str(SWEEP_DIR/'anchor_samples.json'),
    'sweep_metrics':str(sweep_cache_path),
    'pc_label_summary_csv':str(csv_path),
    'pc_label_summary_json':str(json_path),
    'monotonicity_heatmap':str(heatmap_path),
    'duplicate_similarity_plot':str(dup_path),
    'llm_interface_stub':str(LLM_DIR/'llm_control_interface_stub.json'),
    'llm_pc_vector_example':str(LLM_DIR/'pc_vector_stub_example.json'),
}
(OUTPUT_DIR/'output_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('Workflow complete. No VAE training was run.')
print('Authoritative PC labels:', csv_path)
print('Read Cell 9 first: confidence strong/moderate + duplicate_of empty means usable.')
print('Focus first on PC1-PC3. Use PC4-PC5 only if strong/moderate. Keep PC6-PC8 disabled unless clearly strong and non-duplicate.')
print('Manifest:', OUTPUT_DIR/'output_manifest.json')
